# Intersection of Two Arrays

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Arrays, Hash Tables · **Difficulty/Frequency:** Very Common (7/10)

## Concepts

**What this problem is really testing:**
- Recognising that "which values appear in both?" is **set intersection**, not a search problem
- Trading **space for time**: one O(n) preprocessing pass turns an O(m) scan-per-element into an O(1) lookup
- Knowing the **two-pointer** alternative for sorted input, and when it is the better answer

**First-principles primer — what is each piece?**

- **Hash set (`set`)** — an unordered collection with no duplicates. It runs each value through a hash function to compute *where* that value would live, so `x in s` is answered by looking in one place rather than searching. Average **O(1)** membership, and adding a value already present is a silent no-op — which is exactly the "results must be unique" requirement, for free.
- **Average vs. worst case** — hash operations are O(1) *on average*. In the worst case (every value colliding into one bucket) they degrade to O(n). Real-world integer keys never do this by accident, but saying it aloud shows you know the structure, not just the API.
- **Two pointers** — when both inputs are *sorted*, you can walk one index into each, always advancing the one pointing at the smaller value. No extra memory at all.

**The core trade:**

The brute force asks, for each of the n values in `nums1`, "is it anywhere in `nums2`?" — and answers by scanning all m elements. That is n × m work, and it **re-reads `nums2` from scratch every time** even though `nums2` never changes.

Building a set is the fix: one pass over the array, and from then on membership costs O(1). You have spent O(n) memory to remove a factor of m from the running time.

**Why two sets, not one?** The first set (`set1`) answers *"does this value exist in nums1?"*. The second (`result`) enforces *"report each matched value once"*. `nums1=[1,2,2,1], nums2=[2,2]` shows why the second one matters: without it, `2` gets appended twice.

**Simple worked example.** `nums1 = [4, 9, 5]`, `nums2 = [9, 4, 9, 8, 4]`.

Build `set1 = {4, 5, 9}`, then walk `nums2` once:

| value | in `set1`? | `result` after |
|---|---|---|
| 9 | ✅ | `{9}` |
| 4 | ✅ | `{9, 4}` |
| 9 | ✅ | `{9, 4}` — already there, set absorbs it |
| 8 | ❌ | `{9, 4}` |
| 4 | ✅ | `{9, 4}` — absorbed again |

Answer: `[9, 4]` (any order accepted). Eight operations instead of fifteen comparisons — and the gap widens as the arrays grow.

## Problem Statement

Given two integer arrays `nums1` and `nums2`, return an array of their **intersection**. Each element in the result must be **unique**, and the result may be in **any order**.

**Examples**

```python
intersection([1, 2, 2, 1], [2, 2])          # -> [2]
intersection([4, 9, 5], [9, 4, 9, 8, 4])    # -> [9, 4]   (or [4, 9])
```

**Constraints**

- `1 <= len(nums1), len(nums2) <= 1000`
- `0 <= nums1[i], nums2[i] <= 1000`

### Approach 1 — Naive (nested scan)

**Idea:** for each value in `nums1`, scan the whole of `nums2` looking for it. Collect the hits, then strip duplicates at the end.

This is the direct translation of the problem statement into code, and it is correct. The waste is that `nums2` is re-read from the beginning for **every single element** of `nums1`, even though it never changes.

**Time complexity:** **O(n × m)** — at the constraint limit (1000 × 1000) that is a million comparisons.

**Space complexity:** O(k) for the output, where k is the number of distinct shared values.

In [ ]:
from typing import List


def intersection_naive(nums1: List[int], nums2: List[int]) -> List[int]:
    result = []
    for v in nums1:
        if v in result:                  # already reported -> skip (this scan is O(k) too)
            continue
        for w in nums2:                  # re-reads ALL of nums2 for every v
            if v == w:
                result.append(v)
                break
    return result

### Approach 2 — Sorting + two pointers

**Idea:** sort both arrays, then walk one index into each. Because both are ordered, comparing the two current values tells you exactly what to do:

- `a < b` → `a` cannot appear later in `nums2` (everything ahead is bigger) → advance `i`
- `a > b` → symmetric → advance `j`
- `a == b` → a match; record it, then **skip past all copies of that value in both arrays** so it is reported once

Each pointer only ever moves forward, so the walk itself is O(n + m). The sort dominates.

This is worth knowing because it is the **right** answer when the input arrives already sorted (then it is O(n + m) time and O(1) extra space, beating the hash approach on memory), or when the data is too large for a hash set to fit in RAM.

**Time complexity:** O(n log n + m log m) — the sorts; the merge walk is only O(n + m).

**Space complexity:** O(1) extra beyond the output, if sorting in place.

In [ ]:
def intersection_two_pointers(nums1: List[int], nums2: List[int]) -> List[int]:
    a, b = sorted(nums1), sorted(nums2)
    i = j = 0
    result: List[int] = []
    while i < len(a) and j < len(b):
        if a[i] < b[j]:
            i += 1                       # a[i] is too small to ever match - it is gone for good
        elif a[i] > b[j]:
            j += 1
        else:
            result.append(a[i])          # match: record it exactly once...
            val = a[i]
            while i < len(a) and a[i] == val:   # ...then skip every duplicate in BOTH arrays
                i += 1
            while j < len(b) and b[j] == val:
                j += 1
    return result

### Approach 3 — Optimal (hash set)

**Idea:** pay O(n) once to make membership free, then a single pass over the other array answers everything.

Two refinements worth saying out loud:

- **Build the set from the smaller array.** The asymptotic complexity is O(n + m) either way, but the *memory* is O(min(n, m)) instead of O(n) — and that is the difference between fitting in cache and not.
- **The result is a set.** It deduplicates as you go, so `[2, 2]` collapses to `{2}` with no extra pass.

In idiomatic Python this whole approach is `set(nums1) & set(nums2)`; the explicit loop below is written out so the mechanism is visible, and the one-liner is verified against it.

**Time complexity:** **O(n + m)** expected — one pass to build, one pass to probe, each hash operation average O(1).

**Space complexity:** O(min(n, m)) for the set, plus O(k) for the output.

In [ ]:
def intersection(nums1: List[int], nums2: List[int]) -> List[int]:
    # Build the set from the SMALLER array: same big-O, less memory.
    small, large = (nums1, nums2) if len(nums1) <= len(nums2) else (nums2, nums1)
    seen = set(small)                    # O(min(n, m)) space, O(min(n, m)) time
    result = set()
    for v in large:                      # ONE pass; no rescanning
        if v in seen:                    # average O(1)
            result.add(v)                # a set absorbs repeats -> uniqueness for free
    return list(result)


def intersection_pythonic(nums1: List[int], nums2: List[int]) -> List[int]:
    """The same algorithm, written the way you would actually ship it."""
    return list(set(nums1) & set(nums2))

### Follow-up — keep duplicates, matching the minimum count in both arrays

**Idea:** the sibling problem (LeetCode 350). If `nums1` has three 2s and `nums2` has two, the answer contains **two** 2s — `min(3, 2)`.

A set cannot express this, because it has thrown the counts away. A **hash map of counts** (`collections.Counter`) keeps them, and the answer is the *element-wise minimum* of the two count maps — which is exactly what `Counter`'s `&` operator computes.

Iterating the smaller counter matters here for the same reason as before: you only ever look up keys that could possibly match.

**Time complexity:** O(n + m).

**Space complexity:** O(min(n, m)) for the smaller counter.

In [ ]:
from collections import Counter


def intersect_with_duplicates(nums1: List[int], nums2: List[int]) -> List[int]:
    """Each shared value appears min(count in nums1, count in nums2) times."""
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1      # count the smaller array
    counts = Counter(nums1)
    result: List[int] = []
    for v in nums2:
        if counts[v] > 0:                # a Counter returns 0 for missing keys - no KeyError
            result.append(v)
            counts[v] -= 1               # consume one unit of the budget
    return result

## Verification

Check the two worked examples, then confirm all four approaches agree on randomised input, plus the edge cases that separate a correct answer from a plausible one.

In [ ]:
import random

APPROACHES = [intersection_naive, intersection_two_pointers, intersection, intersection_pythonic]

# --- The two examples from the problem statement (order-insensitive) ---
for fn in APPROACHES:
    assert sorted(fn([1, 2, 2, 1], [2, 2])) == [2], fn.__name__
    assert sorted(fn([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9], fn.__name__

# --- Edge cases ---
for fn in APPROACHES:
    assert fn([1, 2, 3], [4, 5, 6]) == [], fn.__name__            # no overlap
    assert sorted(fn([1, 1, 1], [1, 1])) == [1], fn.__name__      # all duplicates -> one value
    assert sorted(fn([7], [7])) == [7], fn.__name__               # single element
    assert fn([1, 2, 3], []) == [], fn.__name__                   # empty second array
    assert fn([], [1, 2, 3]) == [], fn.__name__                   # empty first array
    assert sorted(fn([0, 1000], [1000, 0])) == [0, 1000], fn.__name__   # constraint bounds
    # The result is unique BY CONSTRUCTION, not by luck:
    out = fn([1, 1, 2, 2, 3, 3], [1, 1, 2, 2, 3, 3])
    assert len(out) == len(set(out)) == 3, fn.__name__

# The inputs must not be mutated (two_pointers sorts COPIES, not the caller's lists)
a, b = [3, 1, 2], [2, 3]
for fn in APPROACHES:
    fn(a, b)
    assert a == [3, 1, 2] and b == [2, 3], f"{fn.__name__} mutated its input"

# --- All four agree on randomised data, within the stated constraints ---
random.seed(23)
for _ in range(400):
    n1 = [random.randint(0, 1000) for _ in range(random.randint(0, 40))]
    n2 = [random.randint(0, 1000) for _ in range(random.randint(0, 40))]
    expected = sorted(set(n1) & set(n2))
    for fn in APPROACHES:
        assert sorted(fn(n1, n2)) == expected, (fn.__name__, n1, n2)

# --- Follow-up: duplicates kept at the minimum count ---
assert sorted(intersect_with_duplicates([1, 2, 2, 1], [2, 2])) == [2, 2]
assert sorted(intersect_with_duplicates([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]
assert intersect_with_duplicates([1, 1, 1], [1]) == [1]          # min(3, 1) == 1
assert sorted(intersect_with_duplicates([1, 1, 1], [1, 1])) == [1, 1]   # min(3, 2) == 2
assert intersect_with_duplicates([], [1]) == []

for _ in range(300):
    n1 = [random.randint(0, 20) for _ in range(random.randint(0, 25))]
    n2 = [random.randint(0, 20) for _ in range(random.randint(0, 25))]
    expected = sorted((Counter(n1) & Counter(n2)).elements())     # element-wise minimum
    assert sorted(intersect_with_duplicates(n1, n2)) == expected, (n1, n2)

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Both arrays already sorted.** Skip the sorts and run the two-pointer walk directly: O(n + m) time, **O(1) extra space**. This genuinely beats the hash approach, which always pays O(min(n, m)) memory. Sortedness is a gift — say so when you spot it.
- **One array far larger than the other, and memory is tight.** Sort only the *small* one (O(s log s)) and binary-search each element of the large one: O(l log s) time, O(s) space. Better than hashing the large array when `s ≪ l`. If even the small array does not fit, fall back to an external merge sort of both and a streaming intersection — which never holds more than a buffer of each in memory.
- **Neither array fits in memory.** Sort both on disk, then stream the two-pointer merge. Only one element of each is resident at a time. This is precisely how a database computes a merge-join, and why sorted indexes are so useful.
- **Approximate matching ("within distance d").** Hashing dies here, because near-misses do not collide. Sort one array and, for each value `v` in the other, `bisect` the window `[v-d, v+d]`: O((n + m) log n). For higher dimensions the same idea generalises to a k-d tree or locality-sensitive hashing.
- **Intersecting k arrays, not two.** Intersect the **smallest** array against the next-smallest first, since the running result can only shrink — the same smallest-first ordering trick that makes an inverted index's AND query fast.

## Empirical complexity check

Compare the **nested scan** (Approach 1, O(n × m)) against the **hash set** (Approach 3, O(n + m)) on two arrays that both double in size. Inputs are drawn from a wide value range so overlap stays sparse and the naive version never gets to exit early.

| Growth when n doubles | What it means |
|---|---|
| ~4x | quadratic — both loops grew, so the work grew by 2 × 2 |
| ~2x | linear — one pass over each array, regardless of overlap |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random


def make_arrays(n):
    rng = random.Random(5)
    # A wide value range keeps overlap sparse, so the naive inner loop rarely breaks early.
    return ([rng.randrange(n * 10) for _ in range(n)],
            [rng.randrange(n * 10) for _ in range(n)])


benchmark(
    {"Approach 1 - nested scan O(n*m)": intersection_naive,
     "Approach 2 - sort + two pointers O(n log n)": intersection_two_pointers,
     "Approach 3 - hash set O(n+m)": intersection},
    make_arrays,
    sizes=[500, 1000, 2000, 4000],
    repeats=2,
)

## Patterns learned

- **"Is X in this collection?" asked repeatedly means build a set.** One O(n) pass converts every future membership question from O(n) to O(1). This single move is behind two-sum, deduplication, cycle detection, and most "find the pair/duplicate" problems.
- **A set gives you uniqueness for free.** When the spec says "each element at most once", collecting into a set beats appending to a list and deduplicating afterwards — the constraint is enforced by the data structure, not by extra code.
- **A set forgets counts; a `Counter` remembers them.** The moment the problem says "how many times", you need the map, not the set. Recognising which one the spec demands is half the question.
- **Preprocess the smaller side.** Same big-O, less memory, better cache behaviour — and it signals that you think about constants, not just exponents.
- **Sorted input unlocks two pointers.** If the data is already ordered, the hash set is no longer optimal: two pointers match its time and beat its space. Always check whether sortedness is given before spending memory.
- **Both pointers only move forward.** That is what makes the merge walk O(n + m) rather than O(n × m), and it is the same engine behind merge sort, merge joins, and interval sweeps.